# Variance Reduction Techniques Monte Carlo 1/2

### Black & Scholes

In [1]:
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

### Standard Monte Carlo

In [2]:
def european_mc(S, K, T, r, sigma, n_sim, option_type='call', seed=32):
    np.random.seed(seed)
    sign = 1 if option_type == 'call' else -1

    Z = np.random.standard_normal(n_sim) # Generate random value from N(0, 1)
    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(sign * (S_T - K), 0)

    price = np.exp(-r * T) * payoffs.mean()
    se = np.exp(-r * T) * payoffs.std(ddof=1) / np.sqrt(n_sim)
    return price, se

### Moment Matching

In [3]:
def european_mc_mm(S, K, T, r, sigma, n_sim, option_type='call', seed=32):
    np.random.seed(seed)
    sign = 1 if option_type == 'call' else -1

    Z = np.random.standard_normal(n_sim) # Generate random value from N(0, 1)
    Z_mm = (Z - Z.mean()) / Z.std(ddof=0) # standardization, formula (12.1)
    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_mm)
    payoffs = np.maximum(sign * (S_T - K), 0)

    price = np.exp(-r * T) * payoffs.mean()
    se = np.exp(-r * T) * payoffs.std(ddof=1) / np.sqrt(n_sim)
    return price, se

In [4]:
# Parameters
S, K, T, r, sigma, n_sim = 100.0, 100.0, 1.0, 0.05, 0.20, 10000

In [5]:
# Pricing
bs_price = black_scholes(S, K, T, r, sigma, 'call')
for n_sim in [150, 10000]:
    price_std, se_std = european_mc(S, K, T, r, sigma, n_sim, 'call')
    price_mm, se_mm = european_mc_mm(S, K, T, r, sigma, n_sim, 'call')

    print(f"--- n = {n_sim:} ---")
    print(f"Standard MC       : {price_std:.4f} ± {1.96*se_std:.4f}")
    print(f"Moment Matching MC: {price_mm:.4f} ± {1.96*se_mm:.4f}")
    print(f"Variance reduction: {(se_std/se_mm)**2:.2f}x")

--- n = 150 ---
Standard MC       : 11.9282 ± 2.6396
Moment Matching MC: 10.5891 ± 2.4251
Variance reduction: 1.18x
--- n = 10000 ---
Standard MC       : 10.6350 ± 0.2906
Moment Matching MC: 10.4743 ± 0.2877
Variance reduction: 1.02x


### Antithetic Variates

In [6]:
def european_mc_av(S, K, T, r, sigma, n_sim, option_type='call', seed=32):
    np.random.seed(seed)
    sign = 1 if option_type == 'call' else -1
    n_pairs = n_sim // 2

    Z = np.random.standard_normal(n_pairs)
    S_T_pos = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    S_T_neg = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * (-Z))

    payoff_pos = np.maximum(sign * (S_T_pos - K), 0)
    payoff_neg = np.maximum(sign * (S_T_neg - K), 0)
    pair_avg = (payoff_pos + payoff_neg) / 2 # pair payoff, formula (12.4)

    price = np.exp(-r * T) * pair_avg.mean()
    se = np.exp(-r * T) * pair_avg.std(ddof=1) / np.sqrt(n_pairs)
    return price, se

In [7]:
# Pricing
bs_price = black_scholes(S, K, T, r, sigma, 'call')
price_std, se_std = european_mc(S, K, T, r, sigma, n_sim, 'call')
price_av, se_av = european_mc_av(S, K, T, r, sigma, n_sim, 'call')

print(f"Black-Scholes             : {bs_price:.4f}")
print(f"Standard MC   (n = {n_sim:}) : {price_std:.4f} ± {1.96*se_std:.4f}")
print(f"Antithetic MC (n = {n_sim:}) : {price_av:.4f} ± {1.96*se_av:.4f}")
print(f"Variance reduction factor : {(se_std/se_av)**2:.2f}x")

Black-Scholes             : 10.4506
Standard MC   (n = 10000) : 10.6350 ± 0.2906
Antithetic MC (n = 10000) : 10.6097 ± 0.2056
Variance reduction factor : 2.00x


### Control Variates

In [8]:
def simulate_paths(S, T, r, sigma, n_step, n_sim , seed=32):
    np.random.seed(seed)
    dt = T / n_step
    
    # Generate a matrix of Z ~ N(0, 1) with dimensions (nb of scenarios, nb of steps)
    Z = np.random.standard_normal((n_sim, n_step))
    growth = np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    # Paths matrix dimension (nb of scenarios, nb of steps + 1)
    paths = np.ones((n_sim, n_step + 1)) * S
    paths[:, 1:] = S * np.cumprod(growth, axis=1)
    
    return paths
    
def asian_mc(S, K, T, r, sigma, n_step, n_sim, option_type='call', asian_type='price'):
    paths = simulate_paths(S, T, r, sigma, n_step, n_sim)
    sign = 1 if option_type == 'call' else -1
    S_avg = paths[:, 1:].mean(axis=1)  # Exclude S0
    S_T   = paths[:, -1]

    if asian_type == 'price':
        payoffs = np.maximum(sign * (S_avg - K), 0)
    elif asian_type == 'strike':
        payoffs = np.maximum(sign * (S_T - S_avg), 0)
    
    price  = np.exp(-r * T) * payoffs.mean()
    se     = np.exp(-r * T) * payoffs.std(ddof=1) / np.sqrt(n_sim)
    
    return price, se

#### Control Variates using European Option

In [9]:
def asian_mc_cv(S, K, T, r, sigma, n_step, n_sim, option_type='call'):
    paths = simulate_paths(S, T, r, sigma, n_step, n_sim)
    sign = 1 if option_type == 'call' else -1

    S_arith = paths[:, 1:].mean(axis=1)
    S_T = paths[:, -1]
    X = np.exp(-r * T) * np.maximum(sign * (S_arith - K), 0) # payoff Arithmetic Asian
    Y = np.exp(-r * T) * np.maximum(sign * (S_T - K), 0) # payoff European

    mu_Y = black_scholes(S, K, T, r, sigma, option_type)
    c_star = np.cov(X, Y, ddof=1)[0, 1] / np.var(Y, ddof=1) # formula (#12.12)
    X_cv = X - c_star * (Y - mu_Y) # formula (#12.9)

    price = X_cv.mean()
    se = X_cv.std(ddof=1) / np.sqrt(n_sim)
    return price, se, c_star

#### Control Variates using Geometric Asian Option

In [10]:
def geometric_asian_bs(S, K, T, r, sigma, m, option_type='call'):
    sigma_hat = sigma * np.sqrt((m + 1) * (2 * m + 1) / (6 * m**2))
    b_hat = 0.5 * (r - 0.5 * sigma**2) * (m + 1) / m + 0.5 * sigma_hat**2

    d1 = (np.log(S / K) + (b_hat + 0.5 * sigma_hat**2) * T) / (sigma_hat * np.sqrt(T))
    d2 = d1 - sigma_hat * np.sqrt(T)

    if option_type == 'call':
        return np.exp(-r * T) * (S * np.exp(b_hat * T) * norm.cdf(d1) - K * norm.cdf(d2))
    else:
        return np.exp(-r * T) * (K * norm.cdf(-d2) - S * np.exp(b_hat * T) * norm.cdf(-d1))

def asian_mc_cv_geo(S, K, T, r, sigma, n_step, n_sim, option_type='call'):
    paths = simulate_paths(S, T, r, sigma, n_step, n_sim)
    sign = 1 if option_type == 'call' else -1

    S_arith = paths[:, 1:].mean(axis=1)
    S_geo = np.exp(np.log(paths[:, 1:]).mean(axis=1))  
    X = np.exp(-r * T) * np.maximum(sign * (S_arith - K), 0) # payoff Arithmetic Asian
    Y = np.exp(-r * T) * np.maximum(sign * (S_geo - K), 0) # payoff European

    mu_Y = geometric_asian_bs(S, K, T, r, sigma, n_step, option_type)
    c_star = np.cov(X, Y, ddof=1)[0, 1] / np.var(Y, ddof=1) # formula (#12.12)
    X_cv = X - c_star * (Y - mu_Y) # formula (#12.9)
    
    price = X_cv.mean()
    se = X_cv.std(ddof=1) / np.sqrt(n_sim)
    return price, se, c_star

In [11]:
n_step, n_sim = 252, 10000

In [12]:
# Pricing
price_naive, se_naive = asian_mc(S, K, T, r, sigma, n_step, n_sim, 'call')
price_cv, se_cv, c_star = asian_mc_cv(S, K, T, r, sigma, n_step, n_sim, 'call')
price_cv_geo, se_cv_geo, c_star_geo = asian_mc_cv_geo(S, K, T, r, sigma, n_step, n_sim, 'call')

print(f"Standard MC     (n={n_sim: })             : {price_naive:.4f} ± {1.96*se_naive:.4f}")
print(f"Control Variate, European (n={n_sim: })   : {price_cv:.4f} ± {1.96*se_cv:.4f}   (c*={c_star:.4f})")
print(f"Variance reduction factor, European    : {(se_naive/se_cv)**2:.2f}x")
print(f"Control Variate, Geo Asian (n={n_sim: })  : {price_cv_geo:.4f} ± {1.96*se_cv_geo:.4f}   (c*={c_star_geo:.4f})")
print(f"Variance reduction factor, Geo Asian   : {(se_naive/se_cv_geo)**2:.2f}x")

Standard MC     (n= 10000)             : 5.6898 ± 0.1560
Control Variate, European (n= 10000)   : 5.7599 ± 0.0853   (c*=0.4552)
Variance reduction factor, European    : 3.34x
Control Variate, Geo Asian (n= 10000)  : 5.7805 ± 0.0043   (c*=1.0353)
Variance reduction factor, Geo Asian   : 1291.69x
